In [1]:
import pandas as pd

In [2]:
%run "Main_Preprocessing_1.py"

C:\Users\shibu\OneDrive\Desktop\ALLAN\Projects in Data Science\GDSC\TRIAL for me\Main_Preprocessing_1.py:9: DtypeWarning: Columns (0: Recurrent Gain Loss, 1: Genes in Segment) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv("genomic_features.csv")


Data loaded:
  df shape : (242035, 19)
  df2 shape: (698000, 9)
  df3 shape: (2266, 96)
  df4 shape: (2132, 49)

After MSI and Growth fill:
  MSI missing         : 630
  Growth Prop missing : 0

After tissue descriptor fills:
  GDSC Tissue descriptor 1 missing: 0
  GDSC Tissue descriptor 2 missing: 0

After feature column cleaning — missing counts:
TARGET                                     27872
DRUG_NAME                                      0
TCGA_DESC                                   1067
Microsatellite instability Status (MSI)      630
Growth Properties                              0
GDSC Tissue descriptor 1                       0
GDSC Tissue descriptor 2                       0

TCGA Stage 1 fill results:
  Missing before                             : 1067
  Filled from cancer-type column             : 360
  Filled manually                            : 707
  Filled from Tissue descriptor 2 (unique)  : 0
  Missing after                              : 0

TCGA Stage 2 (OTHER -> PRA

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")


def evaluate_regression(y_true, y_pred, model_name):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))

    print(model_name)
    print(f"RMSE: {rmse:.6f}")
    print(f"MAE: {mae:.6f}")
    print(f"R2: {r2:.6f}")

    return {
        "Model": model_name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }


# Rebuild feature set from df6 after UNCLASSIFIED rows were dropped
target_col = "LN_IC50"

candidate_feature_cols = [
    "DRUG_NAME",
    "TARGET_PATHWAY",
    "TCGA_DESC",
    "Microsatellite instability Status (MSI)",
    "Growth Properties",
    "GDSC Tissue descriptor 1",
    "GDSC Tissue descriptor 2",
    "mutational_burden",
    "ploidy_snp6",
    "ploidy_wes",
]
candidate_feature_cols = [c for c in candidate_feature_cols if c in df6.columns]

X_candidate = df6[candidate_feature_cols].copy()
y = df6[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_candidate,
    y,
    test_size=0.20,
    random_state=42
)

# Target encoding for DRUG_NAME
X_train = X_train.copy()
X_test = X_test.copy()

if "DRUG_NAME" in X_train.columns:
    X_train["DRUG_NAME"] = X_train["DRUG_NAME"].astype("string").fillna("MISSING")
    X_test["DRUG_NAME"] = X_test["DRUG_NAME"].astype("string").fillna("MISSING")

    drug_mean_map = y_train.groupby(X_train["DRUG_NAME"]).mean()
    global_train_mean = float(y_train.mean())

    X_train["DRUG_NAME_target_enc"] = X_train["DRUG_NAME"].map(drug_mean_map).fillna(global_train_mean)
    X_test["DRUG_NAME_target_enc"] = X_test["DRUG_NAME"].map(drug_mean_map).fillna(global_train_mean)

    X_train = X_train.drop(columns=["DRUG_NAME"])
    X_test = X_test.drop(columns=["DRUG_NAME"])

# One-hot encode remaining categorical columns
ohe_cols = [
    "TCGA_DESC",
    "GDSC Tissue descriptor 1",
    "GDSC Tissue descriptor 2",
    "Microsatellite instability Status (MSI)",
    "Growth Properties",
    "TARGET_PATHWAY",
]
ohe_cols = [c for c in ohe_cols if c in X_train.columns]

numeric_cols = [c for c in X_train.columns if c not in ohe_cols]

for col in ohe_cols:
    X_train[col] = X_train[col].astype("string").fillna("MISSING")
    X_test[col] = X_test[col].astype("string").fillna("MISSING")

for col in numeric_cols:
    X_train[col] = pd.to_numeric(X_train[col], errors="coerce")
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

train_medians = X_train[numeric_cols].median()
X_train[numeric_cols] = X_train[numeric_cols].fillna(train_medians)
X_test[numeric_cols] = X_test[numeric_cols].fillna(train_medians)

X_train_cat = pd.get_dummies(X_train[ohe_cols], columns=ohe_cols, dummy_na=False)
X_test_cat = pd.get_dummies(X_test[ohe_cols], columns=ohe_cols, dummy_na=False)

X_train_cat, X_test_cat = X_train_cat.align(X_test_cat, join="left", axis=1, fill_value=0)

X_train_encoded = pd.concat([X_train_cat, X_train[numeric_cols]], axis=1)
X_test_encoded = pd.concat([X_test_cat, X_test[numeric_cols]], axis=1)

# Create safe feature names for XGBoost and LightGBM
feature_map = pd.DataFrame({
    "original_name": X_train_encoded.columns,
    "safe_name": [f"f_{i:04d}" for i in range(X_train_encoded.shape[1])]
})

X_train_model = X_train_encoded.copy()
X_test_model = X_test_encoded.copy()

X_train_model.columns = feature_map["safe_name"].tolist()
X_test_model.columns = feature_map["safe_name"].tolist()

print("X_train_encoded shape:", X_train_encoded.shape)
print("X_test_encoded shape :", X_test_encoded.shape)
print("X_train_model shape  :", X_train_model.shape)
print("X_test_model shape   :", X_test_model.shape)

feature_map.head()

X_train_encoded shape: (193628, 146)
X_test_encoded shape : (48407, 146)
X_train_model shape  : (193628, 146)
X_test_model shape   : (48407, 146)


,original_name,safe_name
0,TCGA_DESC_ACC,f_0000
1,TCGA_DESC_ALL,f_0001
2,TCGA_DESC_BLCA,f_0002
3,TCGA_DESC_BRCA,f_0003
4,TCGA_DESC_CESC,f_0004


In [7]:
# CatBoost Optuna 
def objective_catboost(trial):  # Starting Parameters
    params = {
        "iterations": trial.suggest_int("iterations", 300, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 30),
        "loss_function": "RMSE",
        "eval_metric": "R2",
        "verbose": 0,
        "random_state": 42
    }

    model = CatBoostRegressor(**params) # FOr t
    model.fit(
        X_train_encoded, y_train,
        eval_set=(X_test_encoded, y_test),
        use_best_model=True,
        early_stopping_rounds=50,
        verbose=False
    )

    y_pred = model.predict(X_test_encoded)
    return r2_score(y_test, y_pred)

study_cat = optuna.create_study(direction="maximize", study_name="catboost_tuning_r2") # call the function to run for 30 trial
study_cat.optimize(objective_catboost, n_trials=30, show_progress_bar=True)

print("Best CatBoost params:")
print(study_cat.best_params)
print(f"Best CatBoost R2: {study_cat.best_value:.6f}")

best_cat_model = CatBoostRegressor(   # runs the whole training dataset witht the best parameters found by optuna
    **study_cat.best_params,
    loss_function="RMSE",
    eval_metric="R2",
    verbose=0,
    random_state=42
)

best_cat_model.fit(
    X_train_encoded, y_train,
    eval_set=(X_test_encoded, y_test),
    use_best_model=True,
    early_stopping_rounds=50,
    verbose=False
)

y_pred_cat_optuna = best_cat_model.predict(X_test_encoded)

catboost_optuna_results = evaluate_regression(
    y_test,
    y_pred_cat_optuna,
    model_name="CatBoost Optuna Tuned"
)

catboost_optuna_results_df = pd.DataFrame([catboost_optuna_results])
catboost_optuna_results_df

[I 2026-04-07 20:22:14,244] A new study created in memory with name: catboost_tuning_r2


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-04-07 20:22:19,722] Trial 0 finished with value: 0.7748017973740716 and parameters: {'iterations': 461, 'learning_rate': 0.022421418294358994, 'depth': 5, 'l2_leaf_reg': 0.5362555993254895, 'subsample': 0.606783268992115, 'random_strength': 0.09308831291720969, 'bagging_temperature': 0.5241915123078755, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.7748017973740716.
[I 2026-04-07 20:22:33,780] Trial 1 finished with value: 0.832905498276753 and parameters: {'iterations': 1019, 'learning_rate': 0.13589107237618248, 'depth': 5, 'l2_leaf_reg': 0.07675659863969014, 'subsample': 0.9707848373894634, 'random_strength': 0.002376567905277537, 'bagging_temperature': 0.9106393126022111, 'min_data_in_leaf': 20}. Best is trial 1 with value: 0.832905498276753.
[I 2026-04-07 20:22:43,845] Trial 2 finished with value: 0.7993513727499657 and parameters: {'iterations': 842, 'learning_rate': 0.05305616162501758, 'depth': 4, 'l2_leaf_reg': 1.7624879330108358, 'subsample': 0.958235467511398

,Model,RMSE,MAE,R2
0,CatBoost Optuna Tuned,1.034402,0.771603,0.85975


In [8]:
from optuna.samplers import TPESampler

X_tr_xgb, X_val_xgb, y_tr_xgb, y_val_xgb = train_test_split(
    X_train_model,
    y_train,
    test_size=0.15,
    random_state=42
)

def objective_xgb(trial):
    params = {
        "objective": "reg:squarederror",
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
        "n_estimators": trial.suggest_int("n_estimators", 1400, 2400, step=200),
        "learning_rate": trial.suggest_float("learning_rate", 0.035, 0.075),
        "max_depth": trial.suggest_int("max_depth", 8, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 4),
        "subsample": trial.suggest_float("subsample", 0.75, 0.88),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 0.75),
        "gamma": trial.suggest_float("gamma", 0.20, 0.45),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 0.05, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 4.0, 6.5),
    }

    model = XGBRegressor(**params)
    model.fit(X_tr_xgb, y_tr_xgb, eval_set=[(X_val_xgb, y_val_xgb)], verbose=False)

    val_pred = model.predict(X_val_xgb)
    return r2_score(y_val_xgb, val_pred)

study_xgb = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42)
)

study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)

print("Best XGBoost params:")
print(study_xgb.best_params)
print(f"Best validation R2: {study_xgb.best_value:.6f}")

best_xgb_params = study_xgb.best_params.copy()
best_xgb_params.update({
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist"
})

xgb_final_tuned = XGBRegressor(**best_xgb_params)
xgb_final_tuned.fit(X_train_model, y_train)

y_pred_xgb = xgb_final_tuned.predict(X_test_model)

xgb_optuna_results = evaluate_regression(
    y_test,
    y_pred_xgb,
    "XGBoost Optuna Tuned"
)

xgb_optuna_results_df = pd.DataFrame([xgb_optuna_results])
xgb_optuna_results_df

[I 2026-04-07 20:35:48,644] A new study created in memory with name: no-name-9ef72fe9-bf5b-488c-880c-2148f066801f


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-04-07 20:36:16,186] Trial 0 finished with value: 0.8521337220361418 and parameters: {'n_estimators': 1800, 'learning_rate': 0.07302857225639664, 'max_depth': 10, 'min_child_weight': 3, 'subsample': 0.7702824232575167, 'colsample_bytree': 0.6233991780504303, 'gamma': 0.21452090304204988, 'reg_alpha': 0.021766241123453687, 'reg_lambda': 5.502787529358022}. Best is trial 0 with value: 0.8521337220361418.
[I 2026-04-07 20:36:55,943] Trial 1 finished with value: 0.8546392654122572 and parameters: {'n_estimators': 2200, 'learning_rate': 0.0358233797718321, 'max_depth': 10, 'min_child_weight': 4, 'subsample': 0.7776040843881759, 'colsample_bytree': 0.627273745081065, 'gamma': 0.24585112746335847, 'reg_alpha': 0.0006624310605949989, 'reg_lambda': 5.311891079080595}. Best is trial 1 with value: 0.8546392654122572.
[I 2026-04-07 20:37:22,533] Trial 2 finished with value: 0.8542630581804044 and parameters: {'n_estimators': 1800, 'learning_rate': 0.04664916560792168, 'max_depth': 9, 'min_c

,Model,RMSE,MAE,R2
0,XGBoost Optuna Tuned,1.03161,0.768748,0.860506


In [9]:
from optuna.samplers import TPESampler

X_tr_lgb, X_val_lgb, y_tr_lgb, y_val_lgb = train_test_split(
    X_train_model,
    y_train,
    test_size=0.15,
    random_state=42
)

def objective_lgbm(trial):
    max_depth = trial.suggest_int("max_depth", 10, 13)

    params = {
        "objective": "regression",
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
        "n_estimators": trial.suggest_int("n_estimators", 1500, 2250, step=250),
        "learning_rate": trial.suggest_float("learning_rate", 0.035, 0.07),
        "max_depth": max_depth,
        "num_leaves": trial.suggest_int("num_leaves", 120, 220),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 30, step=5),
        "subsample": trial.suggest_float("subsample", 0.60, 0.85),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.62, 0.80),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.2, 2.5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.2, 1.0),
    }

    model = LGBMRegressor(**params)
    model.fit(
        X_tr_lgb,
        y_tr_lgb,
        eval_set=[(X_val_lgb, y_val_lgb)],
        eval_metric="l2"
    )

    val_pred = model.predict(X_val_lgb)
    return r2_score(y_val_lgb, val_pred)

study_lgbm = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42)
)

study_lgbm.optimize(objective_lgbm, n_trials=30, show_progress_bar=True)

print("Best LightGBM params:")
print(study_lgbm.best_params)
print(f"Best validation R2: {study_lgbm.best_value:.6f}")

best_lgbm_params = study_lgbm.best_params.copy()
best_lgbm_params.update({
    "objective": "regression",
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1
})

lgbm_final_tuned = LGBMRegressor(**best_lgbm_params)
lgbm_final_tuned.fit(X_train_model, y_train)

y_pred_lgbm = lgbm_final_tuned.predict(X_test_model)

lgbm_optuna_results = evaluate_regression(
    y_test,
    y_pred_lgbm,
    "LightGBM Optuna Tuned"
)

lgbm_optuna_results_df = pd.DataFrame([lgbm_optuna_results])
lgbm_optuna_results_df

[I 2026-04-07 20:49:46,084] A new study created in memory with name: no-name-17331c7b-53bc-4773-915c-477551e6da69


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-04-07 20:49:58,372] Trial 0 finished with value: 0.8534662087611822 and parameters: {'max_depth': 11, 'n_estimators': 2250, 'learning_rate': 0.06061978796339919, 'num_leaves': 180, 'min_child_samples': 10, 'subsample': 0.6389986300840507, 'colsample_bytree': 0.6304550501902759, 'reg_alpha': 2.1922051352823506, 'reg_lambda': 0.6808920093945671}. Best is trial 0 with value: 0.8534662087611822.
[I 2026-04-07 20:50:07,786] Trial 1 finished with value: 0.8536037809545018 and parameters: {'max_depth': 12, 'n_estimators': 1500, 'learning_rate': 0.06894684482566982, 'num_leaves': 204, 'min_child_samples': 15, 'subsample': 0.6454562418017751, 'colsample_bytree': 0.6530128117736181, 'reg_alpha': 0.8997571588069366, 'reg_lambda': 0.6198051453057902}. Best is trial 1 with value: 0.8536037809545018.
[I 2026-04-07 20:50:16,130] Trial 2 finished with value: 0.8526923609264805 and parameters: {'max_depth': 11, 'n_estimators': 1750, 'learning_rate': 0.05641485131528329, 'num_leaves': 134, 'min_

,Model,RMSE,MAE,R2
0,LightGBM Optuna Tuned,1.038899,0.774794,0.858528
